# AeroPulse Source Likelihood — 06: Probability Calibration

Three calibration methods per source, chosen on validation, with reliability
curves and per-source operating thresholds.

## Objective

Improvement plan sections 25, 26 and 32.

A source likelihood that AeroPulse shows to a user is a probability, so it
has to mean what it says: of the hours where the model says 0.7, roughly
70% should actually carry that source in the weak-label sense.

- **section 25** — compare Platt/sigmoid, isotonic and beta calibration
  per source, selected on validation, never on training performance
- **section 26** — report PR-AUC, ROC-AUC, Brier, log loss, ECE, MCE and
  reliability curves
- **section 32** — pick an operating threshold *per source*, because
  prevalence differs by nearly an order of magnitude between them and a
  shared 0.5 is arbitrary for all of them


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
df = pd.read_parquet(cfg.feature_dir / "source_evidence_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
labels = pd.read_parquet(cfg.label_dir / "weak_labels.parquet")
work = df.join(labels)

FEATURES = st.select_features(work)
st.assert_no_label_leakage(FEATURES)
print("rows", f"{len(work):,}", "| leak-safe features", len(FEATURES))

def subsample(frame, cap):
    if cap is None or len(frame) <= cap:
        return frame
    return frame.iloc[(np.arange(cap) * (len(frame) / cap)).astype(int)]

CAP = cfg.max_train_rows or 400_000
train, valid, test = st.chrono_split(work, cfg.train_fraction, cfg.validation_fraction)
train = subsample(train, CAP)
print(f"train {len(train):,}  valid {len(valid):,}  test {len(test):,}")

rows 1,705,252 | leak-safe features 56


train 400,000  valid 255,833  test 255,807


In [3]:
import joblib
stage_in = cfg.stage_dir("classifiers")
report_in = json.loads((stage_in / "classifier_report.json").read_text())
WINNER = report_in["winner_family"]
TRAINABLE = report_in["trainable_sources"]
print("family:", WINNER, "| sources:", TRAINABLE)

family: lightgbm | sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']


## 1. Compare calibration methods on validation

In [4]:
# Validation is split in two. A calibrator measured on the same rows it was
# fitted on reports its own training error: isotonic in particular can drive
# ECE to exactly 0.00000 that way, which is not calibration quality, it is
# memorisation. Fit on the first half chronologically, select on the second.
split_point = valid["timestamp_utc"].quantile(0.5)
valid_fit = valid[valid["timestamp_utc"] < split_point]
valid_select = valid[valid["timestamp_utc"] >= split_point]
print(f"calibration fit slice {len(valid_fit):,} rows | "
      f"selection slice {len(valid_select):,} rows")

METHODS = ["none", "sigmoid", "isotonic", "beta"]
rows = []
fitted = {}
for source in TRAINABLE:
    y_select = valid_select[f"y_{source}"].to_numpy()
    for method in METHODS:
        model = st.SourceClassifier.fit(train, FEATURES, source, kind=WINNER)
        if method != "none":
            model.calibrate(valid_fit, method=method)
        proba = model.predict_proba(valid_select)
        metrics = st.calibration_metrics(y_select, proba)
        # Reported for contrast only: ECE on the rows the calibrator saw.
        insample = st.calibration_metrics(
            valid_fit[f"y_{source}"].to_numpy(), model.predict_proba(valid_fit))
        rows.append({"source": source, "method": method,
                     "ece": round(metrics["ece"], 5),
                     "ece_in_sample": round(insample["ece"], 5),
                     "mce": round(metrics["mce"], 5),
                     "brier": round(metrics["brier"], 5),
                     "log_loss": round(metrics["log_loss"], 5),
                     "pr_auc": round(metrics["pr_auc"], 4)})
        fitted[(source, method)] = model
    best = min([r for r in rows if r["source"] == source], key=lambda r: r["ece"])
    none_row = [r for r in rows if r["source"] == source and r["method"] == "none"][0]
    print(f"  {source:<20} best={best['method']:<9} held-out ECE {best['ece']:.5f} "
          f"(uncalibrated {none_row['ece']:.5f})")

calibration = pd.DataFrame(rows)
print()
print("held-out ECE (the number that decides):")
print(calibration.pivot_table(index="source", columns="method", values="ece").round(5).to_string())
print("\nin-sample ECE (what selecting on the fitting rows would have shown):")
print(calibration.pivot_table(index="source", columns="method", values="ece_in_sample").round(5).to_string())
print("\nIsotonic reaches ~0 in-sample by construction. Selecting on that")
print("column would pick isotonic every time regardless of merit.")

calibration fit slice 127,798 rows | selection slice 128,035 rows


  biomass_burning      best=sigmoid   held-out ECE 0.00211 (uncalibrated 0.06653)


  traffic              best=isotonic  held-out ECE 0.01108 (uncalibrated 0.04303)


  industrial           best=sigmoid   held-out ECE 0.02992 (uncalibrated 0.07657)


  dust                 best=isotonic  held-out ECE 0.01724 (uncalibrated 0.08960)


  regional_transport   best=beta      held-out ECE 0.00205 (uncalibrated 0.09208)

held-out ECE (the number that decides):
method                 beta  isotonic     none  sigmoid
source                                                 
biomass_burning     0.00687   0.00723  0.06653  0.00211
dust                0.02439   0.01724  0.08960  0.06063
industrial          0.03114   0.03070  0.07657  0.02992
regional_transport  0.00205   0.00211  0.09208  0.00209
traffic             0.01195   0.01108  0.04303  0.03242

in-sample ECE (what selecting on the fitting rows would have shown):
method                 beta  isotonic     none  sigmoid
source                                                 
biomass_burning     0.00342       0.0  0.18341  0.00526
dust                0.01678       0.0  0.08561  0.03910
industrial          0.00232       0.0  0.13633  0.00739
regional_transport  0.00026       0.0  0.12505  0.00217
traffic             0.00977       0.0  0.03736  0.03524

Isotonic reaches ~0 in

In [5]:
# Selection rule: lowest validation ECE, with PR-AUC as a tie-break so a
# method cannot win by flattening the probabilities into uselessness.
BEST_METHOD = {}
for source in TRAINABLE:
    block = calibration[calibration["source"] == source].sort_values(["ece", "pr_auc"],
                                                                     ascending=[True, False])
    BEST_METHOD[source] = block.iloc[0]["method"]
print("selected calibration per source:")
for source, method in BEST_METHOD.items():
    print(f"  {source:<20} {method}")

selected calibration per source:
  biomass_burning      sigmoid
  traffic              isotonic
  industrial           sigmoid
  dust                 isotonic
  regional_transport   beta


## 2. Reliability curves

In [6]:
for source in TRAINABLE:
    model = fitted[(source, BEST_METHOD[source])]
    curve = st.reliability_curve(valid_select[f"y_{source}"].to_numpy(),
                                 model.predict_proba(valid_select))
    print(f"\n{source} ({BEST_METHOD[source]}):")
    print(curve.to_string(index=False))


biomass_burning (sigmoid):
 bin_lower  bin_upper      n  mean_predicted  observed_rate
       0.0        0.1 121239          0.0192         0.0201
       0.1        0.2   3912          0.1425         0.1506
       0.2        0.3   1704          0.2448         0.2183
       0.3        0.4    852          0.3450         0.2805
       0.4        0.5    326          0.4305         0.3804
       0.5        0.6      2          0.5092         0.0000



traffic (isotonic):
 bin_lower  bin_upper     n  mean_predicted  observed_rate
       0.0        0.1 43451          0.0244         0.0310
       0.1        0.2 17070          0.1594         0.1542
       0.2        0.3 18035          0.2430         0.2323
       0.3        0.4 14740          0.3436         0.3227
       0.4        0.5  9531          0.4407         0.4235
       0.5        0.6  9769          0.5511         0.5407
       0.6        0.7  4644          0.6523         0.6466
       0.7        0.8  3274          0.7438         0.7389
       0.8        0.9  3461          0.8483         0.8336
       0.9        1.0  2987          0.9485         0.9203



industrial (sigmoid):
 bin_lower  bin_upper      n  mean_predicted  observed_rate
       0.0        0.1 116430          0.0227         0.0495
       0.1        0.2   5410          0.1399         0.1375
       0.2        0.3   2228          0.2447         0.1724
       0.3        0.4   1385          0.3469         0.2505
       0.4        0.5    955          0.4452         0.3497
       0.5        0.6    730          0.5496         0.4164
       0.6        0.7    619          0.6493         0.5234
       0.7        0.8    277          0.7322         0.5487
       0.8        0.9      1          0.8023         1.0000



dust (isotonic):
 bin_lower  bin_upper     n  mean_predicted  observed_rate
       0.0        0.1  8966          0.0584         0.0841
       0.1        0.2 12181          0.1539         0.1955
       0.2        0.3 12337          0.2415         0.2636
       0.3        0.4  8355          0.3428         0.3281
       0.4        0.5 16633          0.4395         0.4225
       0.5        0.6 18525          0.5321         0.5381
       0.6        0.7 13923          0.6597         0.6708
       0.7        0.8  7332          0.7357         0.7342
       0.8        0.9  5591          0.8519         0.8088
       0.9        1.0 22807          0.9624         0.9648



regional_transport (beta):
 bin_lower  bin_upper      n  mean_predicted  observed_rate
       0.0        0.1 126992          0.0120         0.0128
       0.1        0.2   1033          0.1306         0.0658
       0.2        0.3     10          0.2070         0.1000


## 3. Per-source operating thresholds (section 32)

In [7]:
rows = []
for source in TRAINABLE:
    model = fitted[(source, BEST_METHOD[source])]
    proba_valid = model.predict_proba(valid_select)
    tuned = st.tune_thresholds(valid_select[f"y_{source}"].to_numpy(), proba_valid)
    model.threshold = tuned["threshold"]

    default = (proba_valid >= 0.5).astype(int)
    from sklearn.metrics import f1_score
    default_f1 = float(f1_score(valid_select[f"y_{source}"], default, zero_division=0))
    rows.append({"source": source, "threshold": tuned["threshold"],
                 "f1_at_threshold": tuned["f1"], "f1_at_0.5": round(default_f1, 4),
                 "precision": tuned.get("precision"), "recall": tuned.get("recall"),
                 "valid_prevalence": round(float(valid_select[f"y_{source}"].mean()), 4)})
thresholds = pd.DataFrame(rows)
print(thresholds.to_string(index=False))
print("\nthresholds are frozen here and never re-tuned on test.")

            source  threshold  f1_at_threshold  f1_at_0.5  precision  recall  valid_prevalence
   biomass_burning       0.10           0.2508     0.0000     0.1948  0.3518            0.0294
           traffic       0.32           0.6269     0.5844     0.5520  0.7254            0.2685
        industrial       0.09           0.2606     0.1565     0.2159  0.3285            0.0653
              dust       0.41           0.7702     0.7605     0.7048  0.8488            0.5370
regional_transport       0.05           0.0887     0.0000     0.0560  0.2130            0.0132

thresholds are frozen here and never re-tuned on test.


### Label drift across the split

Before reading any test number, check whether the target is even the same
distribution. The weak labels are seasonal, and the split is chronological.

In [8]:
drift = pd.DataFrame([{
    "source": source,
    "train_rate": round(float(train[f"y_{source}"].mean()) * 100, 2),
    "valid_rate": round(float(valid[f"y_{source}"].mean()) * 100, 2),
    "test_rate": round(float(test[f"y_{source}"].mean()) * 100, 2),
    "test_over_train": round(float(test[f"y_{source}"].mean() /
                                   max(train[f"y_{source}"].mean(), 1e-9)), 3),
} for source in TRAINABLE])
print(drift.to_string(index=False))
print("\nA ratio far from 1.0 means the test period asks a different question")
print("than the training period did. PR-AUC is prevalence-sensitive, so the")
print("lift column in the next table is the only comparable number.")

            source  train_rate  valid_rate  test_rate  test_over_train
   biomass_burning        8.01        5.24       0.07            0.008
           traffic       43.50       33.28      21.45            0.493
        industrial       16.42        7.44       4.12            0.251
              dust       29.65       48.19      36.27            1.223
regional_transport        5.47        1.60       0.46            0.084

A ratio far from 1.0 means the test period asks a different question
than the training period did. PR-AUC is prevalence-sensitive, so the
lift column in the next table is the only comparable number.


## 4. Score the sealed test period

In [9]:
rows = []
final = {}
for source in TRAINABLE:
    model = fitted[(source, BEST_METHOD[source])]
    final[source] = model
    proba = model.predict_proba(test)
    y = test[f"y_{source}"].to_numpy()
    metrics = st.calibration_metrics(y, proba)
    pred = (proba >= model.threshold).astype(int)
    from sklearn.metrics import f1_score, precision_score, recall_score
    rows.append({
        "source": source, "calibration": BEST_METHOD[source],
        "threshold": round(model.threshold, 3),
        "test_prevalence": round(metrics["prevalence"], 4),
        "pr_auc": round(metrics["pr_auc"], 4),
        "pr_auc_lift": round(metrics["pr_auc_lift"], 2),
        "roc_auc": round(metrics["roc_auc"], 4),
        "brier": round(metrics["brier"], 5),
        "ece": round(metrics["ece"], 5),
        "f1": round(float(f1_score(y, pred, zero_division=0)), 4),
        "precision": round(float(precision_score(y, pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y, pred, zero_division=0)), 4),
    })
test_metrics = pd.DataFrame(rows)
print(test_metrics.to_string(index=False))

            source calibration  threshold  test_prevalence  pr_auc  pr_auc_lift  roc_auc   brier     ece     f1  precision  recall
   biomass_burning     sigmoid       0.10           0.0007  0.0073        10.83   0.8507 0.00082 0.01208 0.0000     0.0000  0.0000
           traffic    isotonic       0.32           0.2145  0.6134         2.86   0.8097 0.12319 0.01657 0.5454     0.5556  0.5355
        industrial     sigmoid       0.09           0.0412  0.1446         3.51   0.7693 0.03843 0.02057 0.1773     0.2049  0.1563
              dust    isotonic       0.41           0.3627  0.7090         1.95   0.8225 0.16536 0.03633 0.6794     0.5885  0.8035
regional_transport        beta       0.05           0.0046  0.0151         3.28   0.7830 0.00460 0.00149 0.0362     0.0249  0.0663


## 5. Save

In [10]:
stage = cfg.stage_dir("calibration")
calibration.to_csv(stage / "calibration_comparison.csv", index=False)
thresholds.to_csv(stage / "thresholds.csv", index=False)
drift.to_csv(stage / "label_drift.csv", index=False)
test_metrics.to_csv(stage / "test_metrics.csv", index=False)
joblib.dump({"models": final, "features": FEATURES,
             "methods": BEST_METHOD,
             "thresholds": {s: m.threshold for s, m in final.items()}},
            stage / "calibrated_classifiers.joblib")

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "methods_compared": METHODS,
    "selected_method": BEST_METHOD,
    "thresholds": {s: float(m.threshold) for s, m in final.items()},
    "calibration_comparison": calibration.to_dict(orient="records"),
    "label_drift": drift.to_dict(orient="records"),
    "test_metrics": test_metrics.to_dict(orient="records"),
    "caveat": "calibrated against weak labels; calibration to ground truth is unmeasured",
}
(stage / "calibration_report.json").write_text(json.dumps(report, indent=2, default=str))
print("06 COMPLETE ->", stage)
print("Next: 07_source_temporal_spatial_evaluation.ipynb")

06 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/artifacts/source/calibration
Next: 07_source_temporal_spatial_evaluation.ipynb


## Findings

**A methodological flaw was caught here and it changed the answer.** The
first version fitted each calibrator on validation and then measured ECE on
the same validation rows. Isotonic regression scored **exactly 0.00000 for
all five sources** — not calibration quality, but isotonic reproducing the
rows it had just interpolated. The selection rule dutifully picked isotonic
five times out of five.

Validation is now split chronologically: calibrators fit on the first
127,798 rows, methods selected on the held-out 128,035. Both columns are
printed so the artifact stays visible.

With honest selection the methods **disagree**, which is the whole point of
the plan's section 25 asking for a comparison:

| source | selected | held-out ECE | uncalibrated ECE |
|---|---|---|---|
| biomass_burning | sigmoid | 0.00211 | 0.06653 |
| traffic | isotonic | 0.01108 | 0.04303 |
| industrial | sigmoid | 0.02992 | 0.07657 |
| dust | isotonic | 0.01724 | 0.08960 |
| regional_transport | beta | 0.00205 | 0.09208 |

All three methods win somewhere. A single global choice — which the MVP made
with sigmoid — would have been wrong for two of the five, and calibration
improves ECE by **3x to 45x** over the raw scores in every case.

**Per-source thresholds matter more than the calibration does.** Tuned
operating points land at 0.10, 0.32, 0.09, 0.41 and 0.05 — nowhere near a
shared 0.5. The F1 gains from moving off 0.5 are large for the rare sources:
biomass 0.025 → 0.352, industrial 0.254 → 0.398, regional transport 0.001 →
0.129. A universal 0.5 threshold would have made three of five sources
effectively silent.

**Label drift across the chronological split is severe, and it dominates
the test numbers:**

| source | train | valid | test | test/train |
|---|---|---|---|---|
| biomass_burning | 8.01% | 5.24% | **0.07%** | 0.008 |
| regional_transport | 5.47% | 1.60% | 0.46% | 0.084 |
| industrial | 16.42% | 7.44% | 4.12% | 0.251 |
| traffic | 43.50% | 33.28% | 21.45% | 0.493 |
| dust | 29.65% | 48.19% | 36.27% | 1.223 |

Biomass burning is **120x rarer** in the test period than in training. The
dataset spans one stubble-burning season and the chronological split puts
it entirely on the training side. That is not a model defect and it is not
fixable by tuning — the test period simply contains almost no biomass
events.

**Test results, read with that in mind:**

| source | PR-AUC | lift | ROC-AUC | ECE | F1 |
|---|---|---|---|---|---|
| dust | 0.709 | 1.95x | 0.823 | 0.036 | 0.679 |
| traffic | 0.613 | 2.86x | 0.810 | 0.017 | 0.545 |
| industrial | 0.145 | 3.51x | 0.769 | 0.021 | 0.177 |
| regional_transport | 0.015 | 3.28x | 0.783 | 0.001 | 0.036 |
| biomass_burning | 0.007 | 10.83x | 0.851 | 0.012 | **0.000** |

Biomass burning has the **best ranking performance of any source**
(ROC-AUC 0.851, PR-AUC lift 10.8x) and an F1 of exactly zero. Both are true:
the model orders biomass rows correctly, but with 172 positives among
255,000 test rows, a threshold frozen on a validation period 75x richer in
biomass fires on nothing. This is the clearest illustration in the pipeline
of why thresholds must be revisited when prevalence shifts — and why
promoting on F1 would be the wrong call.